# 06 — Simulación Monte Carlo del Mundial 2026

**Proyecto:** D10Sformer — MIA305 (UdeSA, 2026)  
**Fase:** 6 — Simulación estocástica del torneo completo, 48 equipos, formato FIFA 2026

## Objetivo

1. Entrenar LogReg (mejor baseline calibrado, log_loss=0.86, ECE=0.024) sobre el corpus completo de selecciones (sin val split — ya validamos en Fase 1).
2. Construir features de cada uno de los 48 equipos al inicio del Mundial.
3. Definir un predictor `predict(team_a, team_b) → P(home, draw, away)` que combina ambos features.
4. Correr **10.000 simulaciones** del torneo completo (grupos + 16avos + ... + final).
5. Agregar resultados: para cada equipo, **P(pasa grupo), P(8avos), P(cuartos), P(semis), P(final), P(campeón)**.
6. Visualizar top-10 candidatos al título + barras por etapa.

## Decisiones técnicas justificadas

- **Sampling estocástico, NUNCA argmax.** Cada partido se muestrea de la distribución predicha. Esto produce probabilidades marginales correctas.
- **Features estáticas (no propagamos ELO/form intra-simulación).** Limitación documentada en el paper.
- **Knockout sin empates.** La masa de draw se reparte 50/50 entre home y away (penales).
- **Goles vía Poisson independiente.** Tasa por equipo derivada de P(win) y total esperado de 2.5 goles.

## Pre-requisito

Subir a Drive los archivos nuevos de Fase 6:
- `src/simulation/__init__.py`, `bracket.py`, `simulator.py`
- `tests/test_simulation.py`
- `notebooks/06_montecarlo_wc2026.ipynb`

---
## 1. Setup

In [ ]:
import sys
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    PROJECT_ROOT = Path('/content/drive/MyDrive/d10sformer-v2')
    DATA_ROOT = Path('/content/drive/MyDrive/d10sformer')
else:
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
    DATA_ROOT = PROJECT_ROOT

sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from paths import ensure_paths, print_paths

paths = ensure_paths(project_root=PROJECT_ROOT, data_root=DATA_ROOT)
print_paths(paths)

# Alias legacy usados en notebooks v1
ROOT = paths.project_root
DATA_PROCESSED = paths.data_processed
CORPUS_DIR = paths.corpus_dir
VOCAB_PATH = paths.vocab_path
CKPT_DIR = paths.checkpoints
CHECKPOINTS_V1 = paths.checkpoints_v1
DATA_RAW = paths.data_raw
DATA_INTERIM = paths.data_interim


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys, json, pickle
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# paths: ROOT ya definido en setup
# paths: sys.path ya configurado

DATA_INTERIM = paths.data_interim
DATA_PROCESSED = paths.data_processed
REPORTS = ROOT / 'reports'
REPORTS.mkdir(exist_ok=True)

from simulation.bracket import (
    WC2026_GROUPS, WC2026_GROUPS_RAW, GROUP_NAMES, SPANISH_TO_ENGLISH,
    to_english, ALL_KNOCKOUT_MATCHES, assert_bracket_consistency,
)
from simulation.simulator import monte_carlo, simulate_tournament

assert_bracket_consistency()
print(f'✓ Bracket WC 2026 consistente (48 equipos, 12 grupos, 32 partidos KO)')
print(f'  Grupos:')
for grp, teams in WC2026_GROUPS.items():
    print(f'    {grp}: {", ".join(teams)}')

---
## 2. Cargar el corpus internacional + computar features rolling al inicio del Mundial

Necesitamos para cada equipo el último valor de ELO, form_pts_5 y recent_goals_5. Estos serán los features con los que predecimos cada partido del Mundial.

In [ ]:
df_int = pd.read_parquet(DATA_INTERIM / 'international_matches_with_elo.parquet')
df_int['date'] = pd.to_datetime(df_int['date'])
print(f'Partidos internacionales: {len(df_int):,}')
print(f'Rango: {df_int.date.min().date()} → {df_int.date.max().date()}')

In [ ]:
# Recompute rolling features cronológicamente (igual que Fase 4a)
from collections import defaultdict
import math

df_sorted = df_int.sort_values('date').reset_index(drop=True).copy()
team_history = defaultdict(list)
WINDOW = 5

def rmean(history, idx):
    recent = history[-WINDOW:]
    if not recent: return math.nan
    return sum(x[idx] for x in recent) / len(recent)

home_form, away_form, home_goals, away_goals = [], [], [], []
for _, row in df_sorted.iterrows():
    h, a = row.home_team, row.away_team
    hs, as_ = row.home_score, row.away_score
    home_form.append(rmean(team_history[h], 0))
    away_form.append(rmean(team_history[a], 0))
    home_goals.append(rmean(team_history[h], 1))
    away_goals.append(rmean(team_history[a], 1))
    if pd.notna(hs) and pd.notna(as_):
        if hs > as_:   h_pts, a_pts = 3, 0
        elif hs < as_: h_pts, a_pts = 0, 3
        else:          h_pts, a_pts = 1, 1
        team_history[h].append((h_pts, float(hs)))
        team_history[a].append((a_pts, float(as_)))

df_sorted['home_form_pts_5']     = home_form
df_sorted['away_form_pts_5']     = away_form
df_sorted['home_recent_goals_5'] = home_goals
df_sorted['away_recent_goals_5'] = away_goals

print(f'✓ Features rolling computadas. Cobertura: ~{100*df_sorted.home_form_pts_5.notna().mean():.1f}%')

In [ ]:
# Para cada equipo del Mundial, obtener su ÚLTIMO state (ELO, form, goles)
WC_TEAMS = sorted({t for grp in WC2026_GROUPS.values() for t in grp})
print(f'Equipos del Mundial 2026: {len(WC_TEAMS)}')

# Última fila donde cada equipo jugó como home o away
def latest_state(team):
    rows = df_sorted[(df_sorted.home_team == team) | (df_sorted.away_team == team)]
    if len(rows) == 0:
        return None
    r = rows.iloc[-1]
    if r.home_team == team:
        return {
            'team': team,
            'elo': float(r.home_elo_after) if pd.notna(r.home_elo_after) else float(r.home_elo_before),
            'form_pts': r.home_form_pts_5,
            'recent_goals': r.home_recent_goals_5,
            'last_date': r.date,
        }
    else:
        return {
            'team': team,
            'elo': float(r.away_elo_after) if pd.notna(r.away_elo_after) else float(r.away_elo_before),
            'form_pts': r.away_form_pts_5,
            'recent_goals': r.away_recent_goals_5,
            'last_date': r.date,
        }

team_features = {}
missing = []
for t in WC_TEAMS:
    st = latest_state(t)
    if st is None:
        missing.append(t)
    else:
        team_features[t] = st

print(f'Encontrados: {len(team_features)} / {len(WC_TEAMS)}')
if missing:
    print(f'⚠ Equipos sin datos: {missing}')

# Para equipos sin features (raros, ej. Curaçao puede ser problema de naming), usar default
DEFAULT_FEATURES = {'elo': 1500.0, 'form_pts': 1.0, 'recent_goals': 1.0}
for t in missing:
    team_features[t] = {'team': t, **DEFAULT_FEATURES, 'last_date': None}
    print(f'  → {t}: usando features default')

# Tabla de ELO inicial
tf_df = pd.DataFrame(team_features.values()).sort_values('elo', ascending=False)
print(f'\n--- Top 10 por ELO al inicio del Mundial ---')
print(tf_df.head(10)[['team', 'elo', 'form_pts', 'recent_goals']].to_string(index=False))

---
## 3. Entrenar LogReg sobre el corpus completo

Usamos el feature matrix de Fase 1 (`src/data/feature_engineering.py`) sobre todo el data desde 2014, sin val/test split. Ya validamos en Fase 1 que LogReg = mejor baseline.

In [ ]:
from data.feature_engineering import build_feature_matrix, get_feature_columns, prepare_xy
from models.baselines import LogisticRegressionBaseline

# build_feature_matrix devuelve UN DataFrame con features + target (no una tupla)
df_full = build_feature_matrix(
    df_sorted, windows_form=(5, 10), h2h_window=5, min_date='2014-01-01',
)
print(f'Feature DF: {df_full.shape}')

# prepare_xy separa features (X) y target (y), con one-hot de categóricas
X_full, y_full = prepare_xy(df_full, onehot_categoricals=True)
print(f'X: {X_full.shape}, y: {y_full.shape}')

In [ ]:
# Entrenar LogReg sobre TODO
logreg = LogisticRegressionBaseline()
logreg.fit(X_full, y_full)
print('✓ LogReg entrenado')
fi = logreg.feature_importance()
print('\nFeature importance (top-15):')
for name, imp in sorted(fi.items(), key=lambda x: -abs(x[1]))[:15]:
    print(f'  {name:<30}  {imp:+.4f}')

---
## 4. Construir el predictor `predict_match(team_a, team_b)`

Toma las features de ambos equipos y devuelve [P(home), P(draw), P(away)].

In [ ]:
FEATURE_COLS = get_feature_columns()
print(f'Columnas que espera LogReg ({len(FEATURE_COLS)}):')
for c in FEATURE_COLS:
    print(f'  - {c}')

In [ ]:
def build_feature_vector(team_a, team_b, venue='neutral'):
    """Construye un DataFrame con UNA fila con las features esperadas por LogReg."""
    fa = team_features.get(team_a, {'elo': 1500, 'form_pts': 1.0, 'recent_goals': 1.0})
    fb = team_features.get(team_b, {'elo': 1500, 'form_pts': 1.0, 'recent_goals': 1.0})
    elo_diff = fa['elo'] - fb['elo']
    raw = {
        'tournament_class': 'world_cup_final',
        'neutral': 1 if venue == 'neutral' else 0,
        'home_elo': fa['elo'], 'away_elo': fb['elo'], 'elo_diff': elo_diff,
        'expected_home_win_prob': 1 / (1 + 10 ** (-elo_diff / 400)),
        'home_rest_days': 7, 'away_rest_days': 7,
        'home_form5_pts': fa['form_pts'] if pd.notna(fa['form_pts']) else 1.0,
        'home_form5_gf':  fa['recent_goals'] if pd.notna(fa['recent_goals']) else 1.0,
        'home_form5_ga': 1.0, 'home_form5_gd': 0.0, 'home_form5_n': 5,
        'away_form5_pts': fb['form_pts'] if pd.notna(fb['form_pts']) else 1.0,
        'away_form5_gf':  fb['recent_goals'] if pd.notna(fb['recent_goals']) else 1.0,
        'away_form5_ga': 1.0, 'away_form5_gd': 0.0, 'away_form5_n': 5,
        'home_form10_pts': fa['form_pts'] if pd.notna(fa['form_pts']) else 1.0,
        'home_form10_gf':  fa['recent_goals'] if pd.notna(fa['recent_goals']) else 1.0,
        'home_form10_ga': 1.0, 'home_form10_gd': 0.0, 'home_form10_n': 10,
        'away_form10_pts': fb['form_pts'] if pd.notna(fb['form_pts']) else 1.0,
        'away_form10_gf':  fb['recent_goals'] if pd.notna(fb['recent_goals']) else 1.0,
        'away_form10_ga': 1.0, 'away_form10_gd': 0.0, 'away_form10_n': 10,
        'h2h_n_matches': 0, 'h2h_home_wins': 0, 'h2h_draws': 0,
        'h2h_away_wins': 0, 'h2h_avg_gd_for_home': 0.0,
    }
    return pd.DataFrame([raw])

# Sanity: test con un partido conocido
x_test = build_feature_vector('Argentina', 'France', venue='neutral')
p_test = logreg.predict_proba(x_test)[0]
print(f'\nArgentina vs France (sanity):')
print(f'  P(home_win) = {p_test[0]:.4f}')
print(f'  P(draw)     = {p_test[1]:.4f}')
print(f'  P(away_win) = {p_test[2]:.4f}')
print(f'  → Sum: {p_test.sum():.4f}')

In [ ]:
def logreg_predictor(team_a, team_b, venue='neutral'):
    """Wrapper que cumple la firma Predictor del simulator."""
    x = build_feature_vector(team_a, team_b, venue=venue)
    probs = logreg.predict_proba(x)[0]
    return probs   # array [p_home, p_draw, p_away]

# Test con varios partidos del WC
test_matches = [
    ('Argentina', 'France'),
    ('Brazil', 'Morocco'),
    ('Mexico', 'South Africa'),
    ('England', 'Croatia'),
    ('Spain', 'Uruguay'),
]
print('=== Predicciones de partidos hipotéticos ===')
print(f'{"home":<15} {"away":<15} {"P(home)":<10} {"P(draw)":<10} {"P(away)":<10}')
for a, b in test_matches:
    p = logreg_predictor(a, b)
    print(f'{a:<15} {b:<15} {p[0]:<10.4f} {p[1]:<10.4f} {p[2]:<10.4f}')

---
## 5. Monte Carlo: 10.000 simulaciones del Mundial

In [ ]:
# Speedup: usar PrecomputedPredictor (Fase 7) — reduce 45min a ~12 seg
from simulation.simulator import PrecomputedPredictor

predictor_fast = PrecomputedPredictor(logreg_predictor, WC_TEAMS, venue='neutral', verbose=True)

import time
t0 = time.time()
agg = monte_carlo(predictor_fast, n_iters=10_000, seed=42, progress=True)
elapsed = time.time() - t0
print(f'\n✓ {agg.n_iters:,} simulaciones en {elapsed:.1f}s ({agg.n_iters/elapsed:.1f} sim/s)')

In [ ]:
df_pred = agg.to_dataframe()
df_pred['Equipo'] = df_pred['team']
df_pred = df_pred[['Equipo', 'P_group_advance', 'P_round_of_16', 'P_quarters', 'P_semis', 'P_final', 'P_champion']]
print('=== Top 16 por probabilidad de campeón ===')
print(df_pred.head(16).to_string(index=False, float_format=lambda x: f'{x:.4f}'))

df_pred.to_csv(REPORTS / 'wc2026_predictions.csv', index=False)
print(f'\n✓ Guardado: {REPORTS / "wc2026_predictions.csv"}')

---
## 6. Visualización: top-15 candidatos al título

In [ ]:
TOP_N = 15
top = df_pred.head(TOP_N).iloc[::-1]   # invertir para barh top-down

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(top['Equipo'], top['P_champion'] * 100, color='#1f77b4')
ax.set_xlabel('Probabilidad de campeón (%)')
ax.set_title(f'WC 2026 — Top {TOP_N} candidatos al título  (n={agg.n_iters:,} simulaciones)')
for i, (_, row) in enumerate(top.iterrows()):
    ax.text(row['P_champion'] * 100 + 0.1, i, f'{row["P_champion"]*100:.1f}%', va='center')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(REPORTS / 'wc2026_champion_probabilities.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Barras agrupadas: probabilidad por etapa para los top-10
TOP = 10
top10 = df_pred.head(TOP)

stages = ['P_group_advance', 'P_round_of_16', 'P_quarters', 'P_semis', 'P_final', 'P_champion']
stage_labels = ['Pasa grupo', 'Octavos', 'Cuartos', 'Semis', 'Final', 'Campeón']

fig, ax = plt.subplots(figsize=(14, 6))
x = np.arange(TOP)
width = 0.13
for i, (col, lbl) in enumerate(zip(stages, stage_labels)):
    offset = (i - 2.5) * width
    ax.bar(x + offset, top10[col] * 100, width, label=lbl)
ax.set_xticks(x)
ax.set_xticklabels(top10['Equipo'], rotation=30, ha='right')
ax.set_ylabel('Probabilidad (%)')
ax.set_title(f'WC 2026 — Probabilidad de alcanzar cada etapa, top-{TOP}')
ax.legend(loc='upper right', ncol=2)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(REPORTS / 'wc2026_stage_probabilities.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 7. Análisis cualitativo: ¿quién enfrenta a quién más probablemente?

In [ ]:
# Para CADA equipo top-10, ver con quién más probable se enfrenta en final
# Esto requiere re-correr 1000 sims y registrar matchups en final
from collections import Counter
import numpy as np

rng = np.random.default_rng(42)
final_matchups = Counter()
champion_finals = defaultdict(Counter)   # champion → Counter de rivals
N_DEEP = 2000

for _ in range(N_DEEP):
    res = simulate_tournament(logreg_predictor, rng)
    if res.champion and res.runner_up:
        pair = tuple(sorted([res.champion, res.runner_up]))
        final_matchups[pair] += 1
        champion_finals[res.champion][res.runner_up] += 1

print('=== Finales más frecuentes (2000 sims) ===')
for pair, count in final_matchups.most_common(10):
    print(f'  {pair[0]:<15} vs {pair[1]:<15}  {count} veces ({100*count/N_DEEP:.1f}%)')

print(f'\n=== Si Argentina llega a la final, ¿contra quién? ===')
if 'Argentina' in champion_finals:
    for rival, count in champion_finals['Argentina'].most_common(5):
        total_arg_finals = sum(champion_finals['Argentina'].values())
        print(f'  vs {rival:<15}  {count} ({100*count/total_arg_finals:.1f}% de las finales con Arg)')
else:
    print('  Argentina no fue campeón en estas 2000 sims')

---
## 8. Diagnóstico: ¿coherente con ELO?

Como sanity check, esperamos que los equipos con mayor ELO inicial sean los favoritos al título.

In [ ]:
# Merge con tabla de ELO
df_pred_elo = df_pred.merge(tf_df[['team', 'elo']], left_on='Equipo', right_on='team').drop(columns='team')
df_pred_elo = df_pred_elo.sort_values('P_champion', ascending=False)

fig, ax = plt.subplots(figsize=(10, 7))
ax.scatter(df_pred_elo['elo'], df_pred_elo['P_champion'] * 100, alpha=0.6, s=80, c='#1f77b4')
for _, row in df_pred_elo.iterrows():
    if row['P_champion'] > 0.02:
        ax.annotate(row['Equipo'], xy=(row['elo'], row['P_champion'] * 100),
                    xytext=(5, 5), textcoords='offset points', fontsize=9)
ax.set_xlabel('ELO inicial')
ax.set_ylabel('P(campeón) (%)')
ax.set_title('Coherencia ELO ↔ probabilidad de campeón')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(REPORTS / 'wc2026_elo_vs_champion.png', dpi=120, bbox_inches='tight')
plt.show()

# Correlación de Spearman para chequear monotonicidad
from scipy.stats import spearmanr
rho, p = spearmanr(df_pred_elo['elo'], df_pred_elo['P_champion'])
print(f'\nCorrelación Spearman ELO ↔ P_champion: ρ = {rho:.3f}  (p={p:.3e})')
print('Esperamos ρ alto (>0.7). Si fuera <0.3 algo está raro.')

---
## 9. Suite de tests

In [ ]:
import subprocess
result = subprocess.run(
    ['python', '-m', 'pytest', str(ROOT / 'tests' / 'test_simulation.py'), '-v', '--tb=short'],
    capture_output=True, text=True,
)
print(result.stdout[-3500:])
if result.stderr:
    print('STDERR:', result.stderr[-800:])
print(f'Exit code: {result.returncode}')

---
## 10. Conclusiones de Fase 6

Llenar al final:

- [ ] Equipos sin datos (fallback default): _____
- [ ] Top 3 candidatos al título:
  1. _____ (___ %)
  2. _____ (___ %)
  3. _____ (___ %)
- [ ] P(Argentina campeón): _____ %
- [ ] Final más probable: _____ vs _____
- [ ] Correlación Spearman ELO ↔ P_champion: _____
- [ ] Tiempo por simulación: _____ ms
- [ ] Tests pasados / total: _____ / 16

**Next:** Fase 7 — pipeline live de actualización durante el Mundial. Cuando los partidos reales se jueguen, fijar resultados y re-simular el bracket restante (las probabilidades cambian dinámicamente).